# QCar AttFuse — Inference Walkthrough

This notebook extends `CP_Fusion_MODELS.ipynb` through cooperative AttFusion, post-fusion detection heads, decoding, and NMS.

**Pipeline:** two calibrated front cameras → per-agent LSS/BEV features → pose warp → AttFusion → classification/regression/direction heads → decode + NMS.

## Phase 0 — Environment and deterministic configuration

In [ ]:
from pathlib import Path
from types import SimpleNamespace
from collections import OrderedDict
import os, json, time
from datetime import datetime
import torch
from torch.utils.data import DataLoader

HEAL_ROOT = Path.cwd()
if not (HEAL_ROOT / 'opencood').is_dir(): HEAL_ROOT = (HEAL_ROOT / '..').resolve()
os.chdir(HEAL_ROOT)
import qcar.patches.patch_1cam_loader
import opencood.hypes_yaml.yaml_utils as yaml_utils
from opencood.data_utils.datasets import build_dataset
from opencood.tools import train_utils
from opencood.utils import eval_utils

assert torch.cuda.is_available(), 'HEAL LSS construction requires CUDA in this checkout.'
DEVICE = torch.device('cuda')
USE_AMP = True
CONFIG_PATH = 'qcar/configs/camera_attfuse_onlyfront.yaml'
hypes = yaml_utils.load_yaml(CONFIG_PATH, SimpleNamespace(model_dir=''))
assert hypes.get('test_dir') is None, 'This dataset is development-only.'
print('Config:', CONFIG_PATH, 'device:', DEVICE)

## Phase 1 — Select weights
The default is the verified OPV2V AttFuse initialization. Set `MODEL_DIR` to a notebook training run to select its single `net_epoch_bestval_at*.pth`, or set `EXACT_CHECKPOINT` to the frozen final-fit checkpoint.

In [ ]:
MODEL_DIR = None  # Example: HEAL_ROOT/'opencood/logs/qcar_attfuse_YYYYMMDD_HHMMSS'
EXACT_CHECKPOINT = None  # Example: HEAL_ROOT/'opencood/logs/qcar_attfuse_final_.../net_epoch12.pth'
CHECKPOINT_PATH = (Path(EXACT_CHECKPOINT) if EXACT_CHECKPOINT is not None
                   else HEAL_ROOT / hypes['_qcar_pretrained_checkpoint'])
if EXACT_CHECKPOINT is None and MODEL_DIR is not None:
    candidates = sorted(Path(MODEL_DIR).glob('net_epoch_bestval_at*.pth'))
    assert len(candidates) == 1, candidates
    CHECKPOINT_PATH = candidates[0]
assert CHECKPOINT_PATH.is_file(), CHECKPOINT_PATH
print('Weights:', CHECKPOINT_PATH)
print('Score threshold:', hypes['postprocess']['target_args']['score_threshold'])
print('NMS threshold:', hypes['postprocess']['nms_thresh'])

## Phase 2 — Build deterministic validation data

In [ ]:
dataset = build_dataset(hypes, visualize=False, train=False)
loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0,
                    collate_fn=dataset.collate_batch_test)
sample = dataset[0]
camera = sample['ego']['input_m2']
print('Frames:', len(dataset))
print('Agents/cameras/image:', tuple(camera['imgs'].shape))
print('Intrinsics:', tuple(camera['intrins'].shape))
print('Post-resize scale:', float(camera['post_rots'][0, 0, 0, 0]))

## Phase 3 — Instantiate the full model, fusion, and post-fusion stack
Extracting modules from the full model avoids creating duplicate randomly initialized components.

In [ ]:
model = train_utils.create_model(hypes)
state = torch.load(CHECKPOINT_PATH, map_location='cpu')
state = state.get('model_state_dict', state)
model.load_state_dict(state, strict=True)
model = model.to(DEVICE).eval()

encoder = model.encoder_m2
bev_backbone = model.backbone_m2
pre_fusion_shrinker = model.shrinker_m2
att_fusion = model.fusion_net
post_fusion_shrinker = getattr(model, 'shrink_conv', None)
post_fusion_heads = OrderedDict([
    ('classification', model.cls_head),
    ('regression', model.reg_head),
    ('direction', model.dir_head),
])
postprocessor = dataset.post_processor
print('Fusion instance:', att_fusion)
print('Post-fusion shrinker:', post_fusion_shrinker, '(None means heads follow AttFusion directly)')
print('Post-fusion heads:', post_fusion_heads)
print('Decoder/NMS:', type(postprocessor).__name__)

## Phase 4 — Run and trace fusion + post-fusion heads
`record_len` tells AttFusion how many agents belong to each scene. `pairwise_t_matrix` becomes a normalized affine warp so the peer feature aligns with ego before attention.

In [ ]:
def shape_of(value):
    if torch.is_tensor(value): return tuple(value.shape)
    if isinstance(value, dict): return {k: shape_of(v) for k, v in value.items()}
    return type(value).__name__

trace = {}
handles = []
stage_modules = [('encoder', encoder), ('BEV backbone', bev_backbone),
                 ('pre-fusion shrinker', pre_fusion_shrinker),
                 ('AttFusion', att_fusion)]
if post_fusion_shrinker is not None:
    stage_modules.append(('post-fusion shrinker', post_fusion_shrinker))
stage_modules.extend((f'post-fusion {name}', module)
                     for name, module in post_fusion_heads.items())
for name, module in stage_modules:
    handles.append(module.register_forward_hook(
        lambda module, inputs, output, name=name: trace.update({name: shape_of(output)})))
batch = next(iter(loader))
batch = train_utils.to_device(batch, DEVICE)
with torch.inference_mode(), torch.cuda.amp.autocast(enabled=USE_AMP):
    output = model(batch['ego'])
for handle in handles: handle.remove()
print('record_len:', batch['ego']['record_len'].tolist())
print('pairwise matrix:', tuple(batch['ego']['pairwise_t_matrix'].shape))
for stage, shape in trace.items(): print(stage, '->', shape)

## Phase 5 — Decode anchors and apply NMS
HEAL expects dictionaries keyed by CAV ID during postprocessing, even though intermediate fusion produces one ego output.

In [ ]:
pred_boxes, pred_scores, gt_boxes = dataset.post_process(batch, {'ego': output})
print('Predictions after NMS:', 0 if pred_boxes is None else len(pred_boxes))
print('Ground-truth boxes:', 0 if gt_boxes is None else len(gt_boxes))
if pred_scores is not None:
    print('Top score:', float(pred_scores.max()))
if pred_boxes is not None:
    print('Finite decoded boxes:', bool(torch.isfinite(pred_boxes).all()))

## Phase 6 — Optional full development-validation inference
This is development evaluation only. Do not call it a final test result.

In [ ]:
RUN_FULL_VALIDATION = True
IOU_THRESHOLDS = (0.2, 0.3, 0.5, 0.7)
if RUN_FULL_VALIDATION:
    stats = {threshold: {'tp': [], 'fp': [], 'score': [], 'gt': 0}
             for threshold in IOU_THRESHOLDS}
    predicted_frames = 0
    inference_seconds = []
    with torch.inference_mode():
        for item in loader:
            item = train_utils.to_device(item, DEVICE)
            torch.cuda.synchronize(); started = time.perf_counter()
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                item_output = model(item['ego'])
            torch.cuda.synchronize(); inference_seconds.append(time.perf_counter()-started)
            boxes, scores, ground_truth = dataset.post_process(item, {'ego': item_output})
            predicted_frames += int(boxes is not None and len(boxes) > 0)
            for threshold in IOU_THRESHOLDS:
                eval_utils.caluclate_tp_fp(boxes, scores, ground_truth, stats, threshold)
    metrics = {}
    for threshold in IOU_THRESHOLDS:
        stat = stats[threshold]
        ap, _, _ = eval_utils.calculate_ap(stats, threshold)
        metrics[str(threshold)] = {'ap': ap, 'gt': stat['gt'],
                                   'tp': sum(stat['tp']), 'fp': sum(stat['fp']),
                                   'recall': sum(stat['tp'])/stat['gt'] if stat['gt'] else None}
    report = {'scope': 'development validation, not final test',
              'checkpoint': str(CHECKPOINT_PATH), 'frames': len(dataset),
              'prediction_frames': predicted_frames, 'amp': USE_AMP,
              'mean_model_seconds': sum(inference_seconds)/len(inference_seconds),
              'metrics': metrics}
    output_dir = HEAL_ROOT/'opencood/logs/qcar_inference_reports'
    output_dir.mkdir(parents=True, exist_ok=True)
    report_path = output_dir/('attfuse_validate_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '.json')
    with open(report_path, 'w') as stream: json.dump(report, stream, indent=2)
    print(json.dumps(report, indent=2)); print('Written:', report_path)
else:
    print('Full validation disabled.')

## Phase 7 — Test policy
`test_dir` is intentionally null. A final unbiased result must use a new independent trajectory after model and thresholds are frozen.